In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)

df = pd.read_parquet('data/yellow_tripdata_2024-01.parquet')
df.shape

(2964624, 19)

In [2]:
df.describe()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
count,2.964624e+06,2964624,2964624,2.824462e+06,2.964624e+06,2.824462e+06,2.964624e+06,2.964624e+06,2.964624e+06,2.964624e+06,2.964624e+06,2.964624e+06,2.964624e+06,2.964624e+06,2.964624e+06,2.964624e+06,2.824462e+06,2.824462e+06
mean,1.754204e+00,2024-01-17 00:46:36.431092,2024-01-17 01:02:13.208130,1.339281e+00,3.652169e+00,2.069359e+00,1.660179e+02,1.651167e+02,1.161271e+00,1.817506e+01,1.451598e+00,4.833823e-01,3.335870e+00,5.270212e-01,9.756319e-01,2.680150e+01,2.256122e+00,1.411611e-01
min,1.000000e+00,2002-12-31 22:59:39,2002-12-31 23:05:41,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,-8.990000e+02,-7.500000e+00,-5.000000e-01,-8.000000e+01,-8.000000e+01,-1.000000e+00,-9.000000e+02,-2.500000e+00,-1.750000e+00
25%,2.000000e+00,2024-01-09 15:59:19.750000,2024-01-09 16:16:23,1.000000e+00,1.000000e+00,1.000000e+00,1.320000e+02,1.140000e+02,1.000000e+00,8.600000e+00,0.000000e+00,5.000000e-01,1.000000e+00,0.000000e+00,1.000000e+00,1.538000e+01,2.500000e+00,0.000000e+00
50%,2.000000e+00,2024-01-17 10:45:37.500000,2024-01-17 11:03:51.500000,1.000000e+00,1.680000e+00,1.000000e+00,1.620000e+02,1.620000e+02,1.000000e+00,1.280000e+01,1.000000e+00,5.000000e-01,2.700000e+00,0.000000e+00,1.000000e+00,2.010000e+01,2.500000e+00,0.000000e+00
75%,2.000000e+00,2024-01-24 18:23:52.250000,2024-01-24 18:40:29,1.000000e+00,3.110000e+00,1.000000e+00,2.340000e+02,2.340000e+02,1.000000e+00,2.050000e+01,2.500000e+00,5.000000e-01,4.120000e+00,0.000000e+00,1.000000e+00,2.856000e+01,2.500000e+00,0.000000e+00
max,6.000000e+00,2024-02-01 00:01:15,2024-02-02 13:56:52,9.000000e+00,3.127223e+05,9.900000e+01,2.650000e+02,2.650000e+02,4.000000e+00,5.000000e+03,1.425000e+01,4.000000e+00,4.280000e+02,1.159200e+02,1.000000e+00,5.000000e+03,2.500000e+00,1.750000e+00
std,4.325902e-01,NaN,NaN,8.502817e-01,2.254626e+02,9.823219e+00,6.362391e+01,6.931535e+01,5.808686e-01,1.894955e+01,1.804102e+00,1.177600e-01,3.896551e+00,2.128310e+00,2.183645e-01,2.338558e+01,8.232747e-01,4.876239e-01


# Intial Observation 

1. The min values for fees and total amount is negative. It could be system error or refunds. Worth investigating further. I'll check how many negative values are there for these amounts and if's not too many, i'll remove those rows and if it's significantly high, i'd look into some approach to fix it, maybe by predicting the price for similar location, distance and time of the day
2. total amount(max) is 5000$ - unrelisticaly expensive for a taxi ride
3. pickup and dropoff datetime(min) backs to 2002 - and our data is just for january 2024 - looks like a data entry error
4. passenger_count(min) is 0 - missing data value 
5. There's only two VendorID - 1 or 2, max 6 seems like a data entry error and i'll remove these
6. trip_distance(max) is roughly 312722 miles, which is unrealistic for this dataset. (Remove it from the dataset)
7. why is payment_type number? (investigate)


In [4]:
financial_cols = ['fare_amount', 'tip_amount', 'tolls_amount', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'extra', 'mta_tax', 'improvement_surcharge']

for col in financial_cols:
    neg_count = (df[col] < 0).sum()
    pct = (neg_count / len(df)) * 100
    print(f"{col}: {neg_count} negative values ({pct:.2f}%)")

fare_amount: 37448 negative values (1.26%)
tip_amount: 102 negative values (0.00%)
tolls_amount: 2035 negative values (0.07%)
total_amount: 35504 negative values (1.20%)
congestion_surcharge: 28825 negative values (0.97%)
Airport_fee: 4921 negative values (0.17%)
extra: 17548 negative values (0.59%)
mta_tax: 34434 negative values (1.16%)
improvement_surcharge: 35502 negative values (1.20%)


On further analysis, i see that the negative amouns account for less than 1.5% of the dataset, which is not much. But something interesting, i'd like to see if actually the negative values in one column corresponds to negative values in other columns. And what percentage of rows are we actually deleting if we remove all the rows with any negative value in it.

In [7]:
# Select only numeric columns automatically
numeric_cols = df.select_dtypes(include='number').columns.tolist()

mask = (df[numeric_cols] < 0).any(axis=1)
total_affected = mask.sum()
pct_affected = (total_affected / len(df)) * 100

print(f"Rows with at least one negative financial value: {total_affected} ({pct_affected:.2f}%)")

Rows with at least one negative financial value: 37578 (1.27%)


I see it's only 1.27% of the rows with negative values. 
I am interesting in two scenerios here:
1. Are these actually refunds or payment reverets?
2. or just data entry error and should be immediately removed?

In [8]:
# Look at a sample of rows with negative fare_amount
df[df['fare_amount'] < 0][['tpep_pickup_datetime', 'trip_distance', 'fare_amount', 'total_amount', 'payment_type']].head(10)

,tpep_pickup_datetime,trip_distance,fare_amount,total_amount,payment_type
99,2024-01-01 00:18:24,2.16,-13.5,-18.50,4
506,2024-01-01 00:04:00,0.01,-31.5,-34.25,2
536,2024-01-01 00:41:42,0.47,-5.8,-10.80,4
552,2024-01-01 00:42:02,5.48,-33.1,-38.10,2
682,2024-01-01 00:24:02,8.74,-47.8,-52.80,4
999,2024-01-01 00:14:22,1.17,-9.3,-14.30,4
1057,2024-01-01 00:45:56,1.57,-11.4,-16.40,4
1195,2024-01-01 00:30:18,9.60,-40.1,-45.10,4
1382,2024-01-01 00:33:28,0.00,-3.0,-8.00,4
1472,2024-01-01 00:30:10,1.58,-11.4,-16.40,4


As there's no transaction id, it is not possible to verify if the negative payments are actually payment reverts. 
And as we see here, for the trip_distance 0, the fare_amount is -3.0$ and i'll conclude here that it's just incorrect entry in the dataset and i'll remove all the rows with the negative values.
My reasons for removal:
1. it's only 1.27% of the dataset, so it won't meaningfully skew our analysis.
2. Across the sample, the trip distance, fare_amount and other numberic based data are inconsistent with the negative numbers

# Investigating the DateTime Problem

In [10]:
# Check trips outside of January 2024
invalid_dates = df[
    (df['tpep_pickup_datetime'].dt.year != 2024) |
    (df['tpep_pickup_datetime'].dt.month != 1)
]

print(f"Rows outside January 2024: {len(invalid_dates)} ({len(invalid_dates)/len(df)*100:.2f}%)")
print(f"\nDate range of invalid rows:")
print(f"Min: {invalid_dates['tpep_pickup_datetime'].min()}")
print(f"Max: {invalid_dates['tpep_pickup_datetime'].max()}")

# Calculate trip duration in minutes
df['trip_duration_mins'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60

print(f"\nTrip duration statistics (minutes):")
print(df['trip_duration_mins'].describe())

Rows outside January 2024: 18 (0.00%)

Date range of invalid rows:
Min: 2002-12-31 22:59:39
Max: 2024-02-01 00:01:15

Trip duration statistics (minutes):
count    2.964624e+06
mean     1.561295e+01
std      3.485105e+01
min     -1.356667e+01
25%      7.150000e+00
50%      1.163333e+01
75%      1.868333e+01
max      9.455400e+03
Name: trip_duration_mins, dtype: float64


1. I see that there are 18 invalid date rows. Better to remove them
2. min trip duration we calculated is -1.35 -- good to see how many rows have negative trip duration
3. Max trip duration of 9455 mintures (approx 158 hours) is also unrelaistic, so good to see how many rows exceed 10 hours

In [11]:
# 1. Negative trip duration
neg_duration = (df['trip_duration_mins'] < 0).sum()
print(f"Negative trip duration: {neg_duration} ({neg_duration/len(df)*100:.2f}%)")

# 2. Zero trip duration
zero_duration = (df['trip_duration_mins'] == 0).sum()
print(f"Zero trip duration: {zero_duration} ({zero_duration/len(df)*100:.2f}%)")

# 3. Trips exceeding 10 hours (600 minutes)
long_trips = (df['trip_duration_mins'] > 600).sum()
print(f"Trips over 10 hours: {long_trips} ({long_trips/len(df)*100:.2f}%)")

Negative trip duration: 56 (0.00%)
Zero trip duration: 814 (0.03%)
Trips over 10 hours: 1651 (0.06%)


1. It's clear that the trip duration can't be non-positive, so we'll remove these rows. It's also negligible amount of rows. 
2. Trips over 10 hours are also likely data error and it only constitutes 0.06% of the dataset, so it wont make a differene by removing them. So I'll remove trips exceeding 600 minutes (10 hours)
3. Rows outside January 2024 (18 rows) will also be removed as they fall outside our analysis window.


In [25]:
#passenger_count investigation
print("Passenger count distribution:")
print(df['passenger_count'].value_counts().sort_index())

over_six_passengers = (df['passenger_count'] > 6).sum()
zero_passengers = (df['passenger_count'] == 0).sum()
total_invalid_passenger_count = over_six_passengers + zero_passengers

print(f"\nPassenger count >  (exceeds 6legal NYC taxi limit): {over_six_passengers}")
print(f"Percentage of trips with invalid passengers {(total_invalid_passenger_count)/len(df)*100:.2f}")

#trip_distance investigation
zero_distance_trips = (df['trip_distance'] == 0).sum()
trips_over_100_miles = (df['trip_distance'] > 100).sum()
percentage_invalid_distance = (zero_distance_trips + trips_over_100_miles)/len(df)*100

print(f"\nZero distance trips: {zero_distance_trips}")
print(f"Trips over 100 miles: {trips_over_100_miles}")
print(f"Percentage of invalid distance trips: {percentage_invalid_distance:.2f}")


Passenger count distribution:
passenger_count
0.0      31465
1.0    2188739
2.0     405103
3.0      91262
4.0      51974
5.0      33506
6.0      22353
7.0          8
8.0         51
9.0          1
Name: count, dtype: int64

Passenger count >  (exceeds 6legal NYC taxi limit): 60
Percentage of trips with invalid passengers 1.06

Zero distance trips: 60371
Trips over 100 miles: 59
Percentage of invalid distance trips: 2.04


In [26]:
# Check overlap between invalid passenger and invalid distance
invalid_passengers = (df['passenger_count'] == 0) | (df['passenger_count'] > 6)
invalid_distance = (df['trip_distance'] == 0) | (df['trip_distance'] > 100)

overlap = (invalid_passengers & invalid_distance).sum()
union = (invalid_passengers | invalid_distance).sum()

print(f"Invalid passengers only: {(invalid_passengers & ~invalid_distance).sum()}")
print(f"Invalid distance only: {(invalid_distance & ~invalid_passengers).sum()}")
print(f"Both invalid: {overlap}")
print(f"Total unique rows affected: {union} ({union/len(df)*100:.2f}%)")

# Are zero distance trips also zero fare?
zero_dist = df[df['trip_distance'] == 0]
print(f"\nOf {len(zero_dist)} zero distance trips:")
print(f"Also have zero fare: {(zero_dist['fare_amount'] == 0).sum()}")
print(f"Have positive fare: {(zero_dist['fare_amount'] > 0).sum()}")

Invalid passengers only: 30725
Invalid distance only: 59630
Both invalid: 800
Total unique rows affected: 91155 (3.07%)

Of 60371 zero distance trips:
Also have zero fare: 419
Have positive fare: 56569


In [22]:
# Look at zero distance trips with positive fare
zero_dist_pos_fare = df[(df['trip_distance'] == 0) & (df['fare_amount'] > 0)]

print(f"Stats for zero distance trips with positive fare:")
print(zero_dist_pos_fare[['trip_distance', 'fare_amount', 'trip_duration_mins', 'passenger_count']].describe())

Stats for zero distance trips with positive fare:
       trip_distance   fare_amount  trip_duration_mins  passenger_count
count        56569.0  56569.000000        56569.000000     33772.000000
mean             0.0     26.671974           10.761316         1.317186
std              0.0     47.053870           29.842729         0.829955
min              0.0      0.010000            0.000000         0.000000
25%              0.0      7.900000            0.250000         1.000000
50%              0.0     16.500000            7.083333         1.000000
75%              0.0     30.360000           15.933333         1.000000
max              0.0   5000.000000         2957.466667         8.000000


## Passenger Count & Trip Distance Investigation

**Passenger Count:**
- 31,465 trips have 0 passengers — impossible for a completed trip
- 55,919 trips exceed the NYC legal taxi limit of 4 passengers (2.95% of dataset)
- Small counts for 7, 8, 9 passengers suggest data entry errors

**Trip Distance:**
- 60,371 trips have zero distance recorded (2.04% of dataset)
- Of these, 56,569 have a positive fare and real duration (median 7 mins, mean fare $26)
- These are likely real trips where GPS failed to record distance — but zero distance corrupts distance-based KPIs
- Only 59 trips exceed 100 miles — clear outliers

**Total rows affected:** 146,767 (4.95%) with minimal overlap between the two issues

**Decisions:**
- Remove rows where passenger_count == 0 or passenger_count > 6 (keeping 5 and 6 as NYC allows larger vehicles on some rate codes)
- Remove zero distance trips — unusable for distance-based analysis regardless of fare
- Remove trips over 100 miles — unrealistic for NYC taxi

In [27]:
# Simulate all cleaning conditions together
clean_mask = (
    # Remove negative financial values
    (df[numeric_cols] >= 0).all(axis=1) &
    # Remove invalid dates
    (df['tpep_pickup_datetime'].dt.year == 2024) &
    (df['tpep_pickup_datetime'].dt.month == 1) &
    # Remove invalid trip duration
    (df['trip_duration_mins'] > 0) &
    (df['trip_duration_mins'] <= 600) &
    # Remove invalid passenger count
    (df['passenger_count'] > 0) &
    (df['passenger_count'] <= 6) &
    # Remove invalid trip distance
    (df['trip_distance'] > 0) &
    (df['trip_distance'] <= 100)
)

clean_rows = clean_mask.sum()
removed = len(df) - clean_rows
print(f"Original rows: {len(df):,}")
print(f"Rows after cleaning: {clean_rows:,}")
print(f"Rows removed: {removed:,} ({removed/len(df)*100:.2f}%)")

Original rows: 2,964,624
Rows after cleaning: 2,722,353
Rows removed: 242,271 (8.17%)


In [28]:
print(df.isnull().sum()[df.isnull().sum() > 0])

passenger_count         140162
RatecodeID              140162
store_and_fwd_flag      140162
congestion_surcharge    140162
Airport_fee             140162
dtype: int64


As the number of rows, for all these columns with null values is exactly the same, i think that they all overlap. Just to be extra safe, i'll confirm it:

In [29]:
# Check if nulls all occur in the same rows
null_rows = df[df['passenger_count'].isnull()]
print(f"Rows where passenger_count is null: {len(null_rows)}")
print(f"Of those, also null in RatecodeID: {null_rows['RatecodeID'].isnull().sum()}")
print(f"Of those, also null in congestion_surcharge: {null_rows['congestion_surcharge'].isnull().sum()}")
print(f"Of those, also null in Airport_fee: {null_rows['Airport_fee'].isnull().sum()}")
print(f"Of those, also null in store_and_fwd_flag: {null_rows['store_and_fwd_flag'].isnull().sum()}")

Rows where passenger_count is null: 140162
Of those, also null in RatecodeID: 140162
Of those, also null in congestion_surcharge: 140162
Of those, also null in Airport_fee: 140162
Of those, also null in store_and_fwd_flag: 140162


As we can see here, the null values across columns overlapp and there are in total 140162 null rows.

# Final explore Analysis

My conclusion:
1. Remove all the null rows
2. Remove all the problematic rows with negative or invalid values

In [31]:
# Check how pandas handles null comparisons
null_passenger_rows = df[df['passenger_count'].isnull()]
print(f"Total null passenger rows: {len(null_passenger_rows)}")
print(f"Of those, fail passenger_count > 0 check: {(null_passenger_rows['passenger_count'] > 0).eq(False).sum()}")
print(f"Of those, also have negative financials: {(null_passenger_rows[numeric_cols] < 0).any(axis=1).sum()}")

Total null passenger rows: 140162
Of those, fail passenger_count > 0 check: 140162
Of those, also have negative financials: 2066


The 140,162 null rows are already caught by the passenger_count > 0 
check — in pandas, NaN > 0 evaluates to False, so null rows 
automatically fail this condition and get removed without needing 
a separate null-handling step.